In [1]:
cd ..

/home/karaman/Documents/bet


In [2]:
from src.utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
from pyspark.sql.functions import pandas_udf
import src.utils.config as config
from src.ingestion.last_activity_generator import generate_last_activity


spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()
players_silver = spark.read.parquet("./data/silver/players")
sessions_silver = spark.read.parquet("./data/silver/sessions")
transactions_silver = spark.read.parquet("./data/silver/transactions")
silver_money_events = spark.read.parquet("./data/silver/money_events")

players_silver = players_silver.drop('player_id')
transactions_silver = transactions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')
sessions_silver = sessions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')
silver_money_events = silver_money_events.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')




Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/10 01:12:24 WARN Utils: Your hostname, karaman-VMware-Virtual-Platform, resolves to a loopback address: 127.0.1.1; using 192.168.1.179 instead (on interface ens33)
26/02/10 01:12:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/10 01:12:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
first_last_activity = generate_last_activity(silver_money_events)


In [ ]:
first_last_activity.show(5)

+----------+----------------+---------------+------------------+-----------------+--------------------+-------------------+
|player_idx|first_event_date|last_event_date|first_session_date|last_session_date|first_financial_date|last_financial_date|
+----------+----------------+---------------+------------------+-----------------+--------------------+-------------------+
|     10959|      2024-02-12|     2024-06-21|        2024-02-12|       2024-06-21|          2024-04-21|         2024-04-21|
|     13723|      2024-01-14|     2024-06-30|        2024-01-14|       2024-06-30|                NULL|               NULL|
|     15371|      2024-06-18|     2024-06-18|              NULL|             NULL|          2024-06-18|         2024-06-18|
|     17048|      2024-03-20|     2024-06-28|        2024-03-20|       2024-06-28|          2024-06-03|         2024-06-03|
|     18295|      2024-03-08|     2024-03-08|              NULL|             NULL|          2024-03-08|         2024-03-08|
+-------

In [8]:
silver_money_events.limit(5).show()

+--------------------+-------------------+----------+-------------+------------------+----------+---------+
|            event_id|           event_ts|event_type|signed_amount| balance_after_txn|player_idx|days_diff|
+--------------------+-------------------+----------+-------------+------------------+----------+---------+
|9a4c4edd-2d3c-46f...|2024-04-09 00:00:00|   session|        23.62|176.64000000000001|      1001|       82|
|3fc30150-5513-475...|2024-05-07 00:00:00|   session|         9.28|185.92000000000002|      1001|       54|
|59a1e615-b82b-432...|2024-05-11 00:00:00|   session|         9.37|195.29000000000002|      1001|       50|
|cabdd03e-dc55-42e...|2024-05-11 00:00:00|   session|        68.76|            264.05|      1001|       50|
|a927b322-e280-45a...|2024-05-27 00:00:00|   session|        78.28|342.33000000000004|      1001|       34|
+--------------------+-------------------+----------+-------------+------------------+----------+---------+



In [9]:
test_date = config_.end_date

In [10]:
silver_money_events = silver_money_events.withColumn('days_diff', F.date_diff(F.lit(test_date), F.col('event_ts')))
sessions_silver = sessions_silver.withColumn('days_diff', F.date_diff(F.lit(test_date), F.col('session_date')))
transactions_silver = transactions_silver.withColumn('days_diff', F.date_diff(F.lit(test_date),F.to_date(F.col('transaction_ts'))))

In [11]:
player_behavior = (sessions_silver
 .filter(F.col('days_diff') <= 30)
 .groupBy('player_idx')
 .agg(
            F.sum(F.when(F.col('days_diff') <= 7, 1).otherwise(0)).alias('num_sessions_7d'),
            F.sum(F.when(F.col('days_diff') <= 7, F.col('signed_amount')).otherwise(0)).alias('net_game_result_7d'),
            F.count('*').alias('num_sessions_30d'),
            F.avg(F.col('session_duration_sec')).cast('int').alias('avg_sessions_duration_30d'),
            F.avg(F.col('total_bet_amount')).alias('avg_bet_amount_30d'),
            F.sum(F.col('signed_amount')).alias('net_game_result_30d'),
 )
                
)
player_behavior.show()

+----------+---------------+------------------+----------------+-------------------------+------------------+-------------------+
|player_idx|num_sessions_7d|net_game_result_7d|num_sessions_30d|avg_sessions_duration_30d|avg_bet_amount_30d|net_game_result_30d|
+----------+---------------+------------------+----------------+-------------------------+------------------+-------------------+
|      3091|              6|442.94999999999993|              24|                     1866|          50.38125| 1610.7599999999998|
|     71648|             10|            278.79|              28|                     1957| 53.59142857142858|            1496.51|
|     86791|              4|             114.2|              24|                     2011|52.742916666666666|            1355.48|
|     32667|              0|               0.0|              10|                     1593| 55.67999999999999|             666.19|
|     15663|              0|               0.0|               2|                     1565|